In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, log_loss, brier_score_loss,
                               classification_report, confusion_matrix,
                               roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt

In [ ]:
train = pd.read_csv('train.csv')

X = train.drop(columns=['client_id', 'default'])
y = train['default']

In [ ]:
# TRAIN/VALIDATION SPLIT
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"Default rate — train: {y_train.mean():.3f}, val: {y_val.mean():.3f}")

In [ ]:
#Feature Groups
categorical_cols = ['SEX', 'EDUCATION', 'MARRIAGE']       # nominal -> one-hot
ordinal_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']  # treat as numeric (already ordered ints)
numerical_cols = ['LIMIT_BAL', 'AGE',
                   'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
                   'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

# Logistic regression needs scaled numeric features + one-hot categoricals
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols + ordinal_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

In [ ]:
#MODEL PIPELINE
logreg = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

logreg.fit(X_train, y_train)

In [ ]:
val_proba = logreg.predict_proba(X_val)[:, 1]   # probability of default
val_pred = logreg.predict(X_val)                 # default 0.5 threshold

In [ ]:
print("\n=== Ranking & Calibration ===")
print(f"AUC-ROC:     {roc_auc_score(y_val, val_proba):.4f}")
print(f"Log Loss:    {log_loss(y_val, val_proba):.4f}")
print(f"Brier Score: {brier_score_loss(y_val, val_proba):.4f}")

print("\n=== Classification Report (threshold=0.5) ===")
print(classification_report(y_val, val_pred, target_names=['No Default', 'Default']))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_val, val_pred)
print(cm)
print(f"(rows=actual, cols=predicted; order: No Default, Default)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(y_val, val_proba)
axes[0].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_val, val_proba):.3f}')
axes[0].plot([0,1],[0,1],'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

prec, rec, _ = precision_recall_curve(y_val, val_proba)
axes[1].plot(rec, prec)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

In [ ]:
feature_names = (numerical_cols + ordinal_cols +
                  list(logreg.named_steps['preprocess']
                       .named_transformers_['cat']
                       .get_feature_names_out(categorical_cols)))

coefs = pd.DataFrame({
    'feature': feature_names,
    'coefficient': logreg.named_steps['clf'].coef_[0]
}).sort_values('coefficient', ascending=False)

print("\n=== Top features increasing default risk ===")
print(coefs.head(10))
print("\n=== Top features decreasing default risk ===")
print(coefs.tail(10))